In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from datetime import datetime
import pandas as pd
from glob import glob

**LSTM:**
- sequence lenght = 30
- features = PCA
- loss = MSE
- optimizer = Adam
- batch size = 64

In [2]:
# load the processed data from the CSV file
loaded_data = pd.read_csv('PCA_processed_climatological_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'PCA_processed_climatological_data.csv'

In [ ]:
#** build the sequences **

# select unique locations based on longitude and latitude
unique_locations = loaded_data[['longitude', 'latitude']].drop_duplicates()

X_sequences = []
y_sequences = []

sequence_length = 30

for _, loc in unique_locations.iterrows():
    loc_data = loaded_data[
        (loaded_data['longitude'] == loc['longitude']) &
        (loaded_data['latitude'] == loc['latitude'])
    ].sort_values(by='elapsed_days')

    features = loc_data[['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6']].values
    target = loc_data['E'].values

    for i in range(len(loc_data) - sequence_length):
        X_seq = features[i:i+sequence_length]
        y_val = target[i+sequence_length]
        X_sequences.append(X_seq)
        y_sequences.append(y_val)

X = np.array(X_sequences)
y = np.array(y_sequences)

In [ ]:
#** build a dataset of shape (n_samples, n_features) **

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(X[0][:5])
print(y[0])

Feature matrix shape: (156252, 30, 6)
Target vector shape: (156252,)
[[-1.08733263 -1.30367916  0.81712897 -0.70372497 -0.51489596 -0.79129969]
 [-0.46048293 -2.71307528  1.40493803  0.05134136  0.41994429 -1.01419211]
 [-0.83721439 -1.74891164  1.62980053 -0.1474909  -0.19433768 -1.1331816 ]
 [-0.90566018 -0.83460412  0.83771119 -0.72184927 -0.35064262 -1.29424872]
 [-1.16611563 -0.20218523  0.3233643  -1.05541314 -0.98714769 -1.16150876]]
-0.85567915


In [ ]:
#** mix the sequences of data **
from sklearn.utils import shuffle

X, y = shuffle(X, y, random_state=42)

print(f"Feature matrix shape after shuffle: {X.shape}")
print(f"Target vector shape after shuffle: {y.shape}")
print(X[0][:5])
print(y[0])

Feature matrix shape after shuffle: (156252, 30, 6)
Target vector shape after shuffle: (156252,)
[[ 2.04267063  1.83741034  1.28269879  0.15353406 -1.43182713 -0.20545752]
 [ 2.38098807  1.32357226  0.45579041 -0.01397663 -1.02923179 -0.05667657]
 [ 2.76650384  0.79649688  0.10760745  0.10039634 -0.20102059 -0.4325268 ]
 [ 3.17929062  0.56892787  0.63210398  0.28048847 -0.44728061 -0.08590704]
 [ 3.06376799  0.40301045 -0.15557128  0.10679341  0.15697895 -0.42027523]]
-1.1966311


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Testing set shape: {X_test.shape}, {y_test.shape}")

Training set shape: (125001, 30, 6), (125001,)
Testing set shape: (31251, 30, 6), (31251,)


In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

# the model learns to predict the next day ET value based on the previous 30 days of data

model = tf.keras.Sequential()

model.add(tf.keras.layers.Input(shape=(sequence_length, 6)))  # input shape (sequence_length, n_features)
model.add(tf.keras.layers.LSTM(128, return_sequences=True))   # return_sequences=True, each of the LSTM cells returns its sequence of 128 outputs. So the output shape is (batch_size, sequence_length, 128)

model.add(tf.keras.layers.LSTM(64, return_sequences=False)) # return_sequences=False, the last LSTM cell returns its last output. So the output shape is (batch_size, 64)
model.add(tf.keras.layers.Dropout(0.1))

model.add(tf.keras.layers.Dense(64, activation='relu'))

model.add(tf.keras.layers.Dense(1))  # predict a single ET value

# compile the model
model.compile(optimizer='adam', loss='mse')
model.summary()

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 128)        │        69,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,753 (479.50 KB)

 Trainable params: 122,753 (479.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import time

start_time = time.time()

history = model.fit(
    X_train, y_train,
    batch_size=64,
    validation_split=0.2,
    epochs=50,
)

end_time = time.time()
print(f"Training time: {end_time - start_time:.2f} seconds")

Epoch 1/75
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - loss: 0.2343 - val_loss: 0.2006
Epoch 2/75
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 18s 8ms/step - loss: 0.1814 - val_loss: 0.1772
Epoch 3/75
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - loss: 0.1678 - val_loss: 0.1629
Epoch 4/75
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - loss: 0.1518 - val_loss: 0.1546
Epoch 5/75
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - loss: 0.1395 - val_loss: 0.1444
Epoch 6/75
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 19s 8ms/step - loss: 0.1293 - val_loss: 0.1354
Epoch 7/75
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - loss: 0.1192 - val_loss: 0.1294
Epoch 8/75
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 23s 9ms/step - loss: 0.1105 - val_loss: 0.1177
Epoch 9/75
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.1029 - val_loss: 0.1061
Epoch 10/75
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - loss: 0.0958 - val_loss: 0.1051
Epoch 11/75
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0892 - val_loss: 0.1027
Epoch 12/75
1563/15

In [ ]:
# save the model
model.save('PCA_et_prediction_model.h5')

**LSTM:**
- sequence lenght = 30
- random missing input features
- loss = MSE
- optimizer = Adam
- batch size = 64

In [32]:
# load the processed data from the CSV file
reduced_dataset = pd.read_csv('processed_climatological_data_with_nan.csv')

In [34]:
#** build the sequences **

unique_locations = reduced_dataset[['longitude', 'latitude']].drop_duplicates()

X_sequences = []
y_sequences = []

sequence_length = 30

feature_cols = ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7']

for _, loc in unique_locations.iterrows():
    loc_data = reduced_dataset[
        (reduced_dataset['longitude'] == loc['longitude']) &
        (reduced_dataset['latitude'] == loc['latitude'])
    ].sort_values(by='elapsed_days')

    features = loc_data[feature_cols].values
    target = loc_data['E'].values

    for i in range(len(loc_data) - sequence_length):
        X_seq = features[i:i+sequence_length]
        y_val = target[i+sequence_length]
        X_sequences.append(X_seq)
        y_sequences.append(y_val)

X = np.array(X_sequences)
y = np.array(y_sequences)

In [35]:
#** build a dataset of shape (n_samples, n_features) **

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(X[0][:5])
print(y[0])

Feature matrix shape: (156252, 30, 7)
Target vector shape: (156252,)
[[-0.82978674 -1.1431502   0.85141805 -0.71261053 -0.52306202 -0.92820815
  -0.80250841]
 [-0.46291794 -2.71383335  1.39816482  0.04726615  0.41452445 -1.01420357
  -0.85581847]
 [-0.83856147 -1.75102817  1.62255607 -0.15012411 -0.2020182  -1.13375323
  -0.2795174 ]
 [-0.90585532 -0.83496617  0.83117273 -0.72312738 -0.35686555 -1.29353678
  -0.11906471]
 [-1.04451891 -0.59217946  0.47970007 -0.83641223 -0.57171213 -1.38216061
  -0.19799387]]
-0.85567915


In [36]:
#** mix the sequences of data **
from sklearn.utils import shuffle

X, y = shuffle(X, y, random_state=42)

print(f"Feature matrix shape after shuffle: {X.shape}")
print(f"Target vector shape after shuffle: {y.shape}")
print(X[0][:5])
print(y[0])

Feature matrix shape after shuffle: (156252, 30, 7)
Target vector shape after shuffle: (156252,)
[[ 2.43321138  1.04696042  1.25065575  0.32883979 -1.12567931  0.16865034
  -0.03325949]
 [ 1.81235432  1.15355816  0.48066518  0.0146109  -1.060755    0.07227551
   0.79456471]
 [ 2.76895747  0.79391619  0.1091791   0.10034533 -0.20226925 -0.43124885
   0.90621694]
 [ 3.18126173  0.56411106  0.63188215  0.28027287 -0.45049244 -0.08414317
   0.59176173]
 [ 3.06594505  0.40139236 -0.15272872  0.10634924  0.15792748 -0.41950574
   1.07964128]]
-1.1966311


In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Testing set shape: {X_test.shape}, {y_test.shape}")

Training set shape: (125001, 30, 7), (125001,)
Testing set shape: (31251, 30, 7), (31251,)


In [38]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

# the model learns to predict the next day ET value based on the previous 30 days of data

model = tf.keras.Sequential()

model.add(tf.keras.layers.Input(shape=(sequence_length, 7)))  # input shape (sequence_length, n_features)
model.add(tf.keras.layers.LSTM(128, return_sequences=True))   # return_sequences=True, each of the LSTM cells returns its sequence of 128 outputs. So the output shape is (batch_size, sequence_length, 128)

model.add(tf.keras.layers.LSTM(64, return_sequences=False)) # return_sequences=False, the last LSTM cell returns its last output. So the output shape is (batch_size, 64)
model.add(tf.keras.layers.Dropout(0.1))

model.add(tf.keras.layers.Dense(64, activation='relu'))

model.add(tf.keras.layers.Dense(1))  # predict a single ET value

model.compile(optimizer='adam', loss='mse')
model.summary()

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_9 (LSTM)                   │ (None, 30, 128)        │        69,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_10 (LSTM)                  │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 123,265 (481.50 KB)

 Trainable params: 123,265 (481.50 KB)

 Non-trainable params: 0 (0.00 B)

In [39]:
import time

start_time = time.time()

history = model.fit(
    X_train, y_train,
    batch_size=64,
    validation_split=0.2,
    epochs=50,
)

end_time = time.time()
print(f"Training time: {end_time - start_time:.2f} seconds")

Epoch 1/50
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - loss: 0.2350 - val_loss: 0.1900
Epoch 2/50
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.1798 - val_loss: 0.1598
Epoch 3/50
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.1567 - val_loss: 0.1571
Epoch 4/50
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - loss: 0.1450 - val_loss: 0.1433
Epoch 5/50
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.1322 - val_loss: 0.1396
Epoch 6/50
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - loss: 0.1218 - val_loss: 0.1236
Epoch 7/50
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - loss: 0.1125 - val_loss: 0.1169
Epoch 8/50
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.1057 - val_loss: 0.1204
Epoch 9/50
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - loss: 0.0990 - val_loss: 0.1015
Epoch 10/50
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - loss: 0.0891 - val_loss: 0.1050
Epoch 11/50
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - loss: 0.0882 - val_loss: 0.0968
Epoch 12/50
1563/15

In [41]:
# save the model
model.save('et_prediction_model_nanValues.h5')